# 다른 선수의 무릎 통과 프레임에서 복사뼈 판독하기

동작 시점을 무릎 통과(`transition`)로 맞춘 검증 선수 세 명을 봅니다. 같은 선수의 가까운 복사뼈를 원본에서 먼저 표시하고, YOLO26n-pose가 찍은 발목과의 세로 차이를 원본 픽셀 단위로 확인합니다.

**Run All**을 누른 뒤 각 탭에서 선수 기준 왼쪽·오른쪽을 확인하세요. 이번에는 **발목 하나만** 판독합니다. 복사뼈 위치를 짚을 수 있으면 클릭하고, 윤곽으로 짐작했다면 **가림 추정점으로 변경**을 누르세요. 가려서 위치를 정할 수 없거나 확신이 없으면 **가려서 판독 불가** 또는 **위치 불확실**을 선택하세요. 모두 한 프레임의 판독 완료로 기록됩니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

from wlcv.annotation.pose_points import load_points, pose_point_editor, pose_point_path
from wlcv.annotation.storage import load_events
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.ankle import review_ankle
from wlcv.evaluation.plots import predict_rgb

video_ids = ["a05_sn_f_01", "b03_sn_f_01", "c08_sn_f_01"]
split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id", "role"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id").loc[video_ids].copy()
videos["frame_index"] = [load_events(ANNOTATIONS, video, pass_number=1)["transition"]
                         for video in videos.index]
display(videos[["near_side", "frame_index"]])

In [ ]:
editors = []
for video, info in videos.iterrows():
    editor = pose_point_editor(VIDEOS / info.filename, info.frame_index, info.person_id,
                               info.near_side, split=info.role, joints=("발목",))
    editors.append(editor)

tabs = widgets.Tab(children=editors)
for n, (video, info) in enumerate(videos.iterrows()):
    tabs.set_title(n, f"{video} · {info.frame_index}번")
display(tabs)

## 세 장의 판독을 저장한 뒤 아래 셀만 다시 실행하기

같은 사전학습 가중치, CPU, 입력 크기 640, 사람 박스 기준 0.25를 사용합니다. 발목점 신뢰도 0.50 미만은 **저신뢰**로 표에 표시합니다. 모델 좌표가 있고 사람이 직접 표시한 경우에 `Δy = 수동 y − 모델 y`를 계산합니다. 양수는 모델점이 수동점보다 화면 위에 있다는 뜻입니다. 추정점의 세로 차이는 참고 열에만 남기고 판독 불가·미검출에는 거리값을 비워 둡니다.

세 장은 각각 원본 크기와 촬영 배율이 다릅니다. 방향이 반복되는지를 먼저 보세요. 사람이 여러 명이면 번호가 붙은 후보 그림에서 수행자를 고른 뒤 아래 셀의 `person_choices`에 해당 영상 ID와 번호를 넣으세요.

In [ ]:
person_choices = {}  # 후보가 여럿일 때 사용. 예: {"a05_sn_f_01": 1}
readings = {}
for video, info in videos.iterrows():
    path = pose_point_path(ANNOTATIONS, video, info.frame_index)
    record = load_points(path).get("발목")
    if record is not None:
        if (record["side"] != info.near_side or record["split"] != "validation"
                or record["video_id"] != video or int(record["frame_index"]) != info.frame_index
                or record["person_id"] != info.person_id):
            raise ValueError(f"{video}의 저장된 발목과 검증 영상 정보를 확인하세요.")
        readings[video] = record

pending = [video for video in videos.index if video not in readings]
if pending:
    print("판독할 영상:", ", ".join(pending))
else:
    model = YOLO(str(ROOT / "models/yolo26n-pose.pt"))
    rows = []
    for video in tqdm(videos.index, desc="발목 비교", unit="장"):
        info = videos.loc[video]
        rgb = read_rgb(VIDEOS / info.filename, info.frame_index)
        result = predict_rgb(model, rgb, imgsz=640, conf=0.25, device="cpu").cpu().numpy()
        row, fig = review_ankle(rgb, result, readings[video], title=f"{video} / {info.frame_index}",
                                person_number=person_choices.get(video))
        rows.append({"영상": video, "프레임": info.frame_index, "가까운 쪽": info.near_side, **row})
        plt.show()

    table = pd.DataFrame(rows)
    display(table.style.format(precision=2, na_rep="—"))
    dy = table["Δy (px)"].dropna()
    if len(dy):
        print(f"직접 표시와 모델 좌표가 모두 있는 {len(dy)}장 중 모델점이 더 위에 있는 프레임은 {(dy > 0).sum()}장입니다.")
    else:
        print("방향을 비교할 수 있는 직접 표시점이 없습니다.")
    output_dir = ROOT / "outputs/pose_baseline"
    output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir / "transition_ankles.csv", index=False)